<a href="https://colab.research.google.com/github/MawaSC/Quantum-Simulation-on-IQM-Resonance/blob/main/07_advanced_error_mitigation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

1. Postselect on the conserved particle number

The Fermi–Hubbard model conserves the total number of particles. Postselection uses this property by keeping only measurement outcomes with the expected number of particles.

Noise can produce outcomes that do not satisfy this condition, so removing those results may help reduce the effect of some errors. However, it also means throwing away measurement shots, leaving us with fewer results to work with. It can also introduce bias.

We record the fraction of shots that remain and compare the results before and after postselection against the same reference. We will use these results to determine whether postselection actually improves the accuracy.

In [ ]:
import matplotlib.pyplot as plt

# 1. Define table data

headers = ["Trotter Step (dt)",
    "Total Circuit Depth",
    "Two-Qubit Circuit Depth"]

data = [
    [
        f"{row['dt']:.2f}",
        int(row["Total Circuit Depth"]),
        int(row["Two-Qubit Circuit Depth"])
    ]
    for _, row in depth_df.iterrows()
]


# 2. Set up figure canvas

fig, ax = plt.subplots(figsize=(7, 3.5), dpi=300)
ax.axis('off')


# 3. Add title

ax.set_title(
    "Circuit Depth at Final Time t = 3",
    fontsize=14.5,
    #fontweight='bold',
    #pad=20
)


# 4. Create matplotlib table

table = ax.table(
    cellText=data,
    colLabels=headers,
    cellLoc='center',
    loc='center'
)


# 5. Scale & format text

table.auto_set_font_size(False)
table.set_fontsize(11)
table.scale(1.2, 2.2)


# 6. Apply custom styling

header_color = '#1f4e78'
zebra_color = '#f2f6fa'
border_color = '#d0d7de'

for (row, col), cell in table.get_celld().items():

    cell.set_edgecolor(border_color)
    cell.set_linewidth(1)

    if row == 0:
        # Header
        cell.set_facecolor(header_color)
        cell.get_text().set_color('white')
        cell.get_text().set_weight('bold')
        cell.get_text().set_fontsize(12)

    else:
        # Alternating row colors
        if row % 2 == 0:
            cell.set_facecolor(zebra_color)
        else:
            cell.set_facecolor('#ffffff')

        cell.get_text().set_color('#222222')


# 7. Adjust layout

plt.tight_layout()


# 8. Save as PNG and PDF

plt.savefig(
    'circuit_depth_t3.png',
    bbox_inches='tight',
    dpi=300
)

plt.savefig(
    'circuit_depth_t3.pdf',
    bbox_inches='tight',
    dpi=300
)

plt.show()

In [ ]:
import matplotlib.gridspec as gridspec
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

# ==============================================================================
# 1. PREPARE CIRCUIT DEPTH DATA (Simplified for t = 3.0)
# ==============================================================================
dt_values = [0.05, 0.10, 0.15, 0.20, 0.50]
summary = []

for dt in dt_values:
    n_max = int(3.0 / dt)
    qv_final = build_circuit(dt, n_max, init_occ)
    total_depth_final, two_qubit_depth_final = circuit_depths(qv_final)

    summary.append({
        "dt": dt,
        "Target Time (t)": 3.0,
        "Trotter Steps (N)": n_max,
        "Total Circuit Depth": total_depth_final,
        "Two-Qubit Circuit Depth": two_qubit_depth_final
    })

depth_df = pd.DataFrame(summary)

# ==============================================================================
# 2. CREATE UNIFIED FIGURE LAYOUT WITH PADDED MARGINS
# ==============================================================================
# Increased figure width slightly to 17 to give the right side extra breathing room
fig = plt.figure(figsize=(17, 11), dpi=300)

# GridSpec with custom margins to shift right elements inward
gs = gridspec.GridSpec(
    2, 2,
    height_ratios=[1.8, 1.0],
    hspace=0.35,
    wspace=0.28,
    left=0.06,   # Extra space on left
    right=0.93,  # Prevents right-side clipping
    top=0.93,
    bottom=0.05
)

# ------------------------------------------------------------------------------
# TOP LEFT: Néel State Quench Dynamics (Nested 2x2 Subgrid)
# ------------------------------------------------------------------------------
gs_top_left = gridspec.GridSpecFromSubplotSpec(
    2, 2, subplot_spec=gs[0, 0], hspace=0.32, wspace=0.25
)

axes_neel = [
    fig.add_subplot(gs_top_left[0, 0]),
    fig.add_subplot(gs_top_left[0, 1]),
    fig.add_subplot(gs_top_left[1, 0]),
    fig.add_subplot(gs_top_left[1, 1])
]

for i, ax in enumerate(axes_neel):
    ax.plot(times, nup_q[:, i], 'o-', ms=3.5, color='crimson', label=r'$\langle n_{\uparrow}\rangle$ Sim')
    ax.plot(times, ndn_q[:, i], 's-', ms=3.5, color='royalblue', label=r'$\langle n_{\downarrow}\rangle$ Sim')
    ax.plot(times, nup_e[:, i], '--', color='crimson', alpha=0.55, label='Exact')
    ax.plot(times, ndn_e[:, i], '--', color='royalblue', alpha=0.55)
    ax.plot(hardware_times, nup_hardware_history[:, i], 'o-', ms=3.5, linewidth=1.5, color='lightgreen', markeredgecolor='darkgreen', label=r'$\langle n_{\uparrow}\rangle$ HW')
    ax.plot(hardware_times, ndn_hardware_history[:, i], 's-', ms=3.5, linewidth=1.5, color='darkgreen', label=r'$\langle n_{\downarrow}\rangle$ HW')

    ax.set_title(f"Site {i}", fontsize=10, fontweight='bold')
    ax.grid(alpha=0.3)
    ax.set_ylim(-0.05, 1.05)

    if i in [2, 3]:
        ax.set_xlabel(r"Time $t/t_h^{-1}$", fontsize=9)
    if i in [0, 2]:
        ax.set_ylabel("Occupation", fontsize=9)

axes_neel[0].legend(fontsize=7, loc='lower right')

# ------------------------------------------------------------------------------
# TOP RIGHT: Total Occupation Comparison at t = 3.0
# ------------------------------------------------------------------------------
ax_comp = fig.add_subplot(gs[0, 1])

ax_comp.plot(sites, ideal_total, 'o-', linewidth=2, markersize=6, label="Ideal Simulation")
ax_comp.errorbar(sites, mean_total, yerr=std_total, fmt='s--', capsize=4, linewidth=2, markersize=6, label="Depolarizing Noise (mean ± SD)")
ax_comp.errorbar(sites, mean_hw_total, yerr=std_hw_total, fmt='^-', capsize=4, linewidth=2, markersize=6, label="IQM Hardware (mean ± SD)")

ax_comp.set_xlabel("Lattice Site", fontsize=10, fontweight='bold')
ax_comp.set_ylabel("Total Occupation", fontsize=10, fontweight='bold')
ax_comp.set_title("Total Occupation Comparison ($t = 3.0$)", fontsize=11, fontweight='bold')
ax_comp.set_xticks(sites)
ax_comp.grid(True, linestyle='--', alpha=0.5)
ax_comp.legend(frameon=True, facecolor='white', framealpha=0.9, fontsize=9)

# ------------------------------------------------------------------------------
# BOTTOM: Circuit Depth Scaling Table (Contrained Width)
# ------------------------------------------------------------------------------
ax_table = fig.add_subplot(gs[1, :])
ax_table.axis('off')

table_data = ax_table.table(
    cellText=depth_df.values,
    colLabels=depth_df.columns,
    cellLoc='center',
    loc='center',
    colWidths=[0.12, 0.15, 0.18, 0.22, 0.25]  # Explicit column widths prevent rightward spill
)

table_data.auto_set_font_size(False)
table_data.set_fontsize(9.5)
table_data.scale(0.9, 1.8)

header_color = '#1f4e78'
zebra_color = '#f2f6fa'

for (row, col), cell in table_data.get_celld().items():
    cell.set_edgecolor('#d0d7de')
    if row == 0:
        cell.set_facecolor(header_color)
        cell.get_text().set_color('white')
        cell.get_text().set_weight('bold')
    else:
        cell.set_facecolor(zebra_color if row % 2 == 0 else '#ffffff')
        cell.get_text().set_color('#222222')

ax_table.set_title(
    "Circuit Depth Scaling vs. Trotter Step Size ($\Delta t$) at Target Time $t = 3.0$",
    fontsize=11,
    fontweight='bold',
    pad=12
)

# ==============================================================================
# 3. SAVE WITH BOUNDING BOX OVERRIDE
# ==============================================================================
plt.savefig(
    "fermi_hubbard_unified_summary_page.png",
    bbox_inches='tight',
    pad_inches=0.2,  # Adds explicit outer padding to PDF rendering
    dpi=300
)

plt.savefig(
    "fermi_hubbard_unified_summary_page.pdf",
    bbox_inches='tight',
    pad_inches=0.2,
    dpi=300
)

plt.show()

In [ ]:
import matplotlib.gridspec as gridspec
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

# ==============================================================================
# 1. PREPARE CIRCUIT DEPTH DATA (Simplified for t = 3.0)
# ==============================================================================
dt_values = [0.05, 0.10, 0.15, 0.20, 0.50]
summary = []

for dt in dt_values:
    n_max = int(3.0 / dt)
    qv_final = build_circuit(dt, n_max, init_occ)
    total_depth_final, two_qubit_depth_final = circuit_depths(qv_final)

    summary.append({
        "dt": dt,
        "Target Time (t)": 3.0,
        "Trotter Steps (N)": n_max,
        "Total Circuit Depth": total_depth_final,
        "Two-Qubit Circuit Depth": two_qubit_depth_final
    })

depth_df = pd.DataFrame(summary)

# ==============================================================================
# 2. CREATE UNIFIED FIGURE LAYOUT WITH PADDED MARGINS
# ==============================================================================
# Increased figure width slightly to 17 to give the right side extra breathing room
fig = plt.figure(figsize=(17, 11), dpi=300)

# GridSpec with custom margins to shift right elements inward
gs = gridspec.GridSpec(
    2, 2,
    height_ratios=[1.8, 1.0],
    hspace=0.35,
    wspace=0.28,
    left=0.06,   # Extra space on left
    right=0.93,  # Prevents right-side clipping
    top=0.93,
    bottom=0.05
)

# ------------------------------------------------------------------------------
# TOP LEFT: Néel State Quench Dynamics (Nested 2x2 Subgrid)
# ------------------------------------------------------------------------------
ax_neel_title = fig.add_subplot(gs[0, 0])
ax_neel_title.axis('off')
ax_neel_title.set_title(
    "1D Fermi-Hubbard Quench from a Neel State",
    fontsize=11,
    fontweight='bold',
    pad=15
)

gs_top_left = gridspec.GridSpecFromSubplotSpec(
    2, 2, subplot_spec=gs[0, 0], hspace=0.32, wspace=0.25
)

axes_neel = [
    fig.add_subplot(gs_top_left[0, 0]),
    fig.add_subplot(gs_top_left[0, 1]),
    fig.add_subplot(gs_top_left[1, 0]),
    fig.add_subplot(gs_top_left[1, 1])
]

for i, ax in enumerate(axes_neel):
    ax.plot(times, nup_q[:, i], 'o-', ms=3.5, color='crimson', label=r'$\langle n_{\uparrow}\rangle$ Sim')
    ax.plot(times, ndn_q[:, i], 's-', ms=3.5, color='royalblue', label=r'$\langle n_{\downarrow}\rangle$ Sim')
    ax.plot(times, nup_e[:, i], '--', color='crimson', alpha=0.55, label='Exact')
    ax.plot(times, ndn_e[:, i], '--', color='royalblue', alpha=0.55)

    ax.set_title(f"Site {i}", fontsize=10, fontweight='bold')
    ax.grid(alpha=0.3)
    ax.set_ylim(-0.05, 1.05)

    if i in [2, 3]:
        ax.set_xlabel(r"Time $t/t_h^{-1}$", fontsize=9)
    if i in [0, 2]:
        ax.set_ylabel("Occupation", fontsize=9)

axes_neel[0].legend(fontsize=7, loc='lower right')

# ------------------------------------------------------------------------------
# TOP RIGHT: Total Occupation Comparison at t = 3.0
# ------------------------------------------------------------------------------
ax_comp = fig.add_subplot(gs[0, 1])

ax_comp.plot(sites, ideal_total, 'o-', linewidth=2, markersize=6, label="Ideal Simulation")
ax_comp.errorbar(sites, mean_total, yerr=std_total, fmt='s--', capsize=4, linewidth=2, markersize=6, label="Depolarizing Noise (mean ± SD)")
ax_comp.errorbar(sites, mean_hw_total, yerr=std_hw_total, fmt='^-', capsize=4, linewidth=2, markersize=6, label="IQM Hardware (mean ± SD)")

ax_comp.set_xlabel("Lattice Site", fontsize=10, fontweight='bold')
ax_comp.set_ylabel("Total Occupation", fontsize=10, fontweight='bold')
ax_comp.set_title("Total Occupation Comparison ($t = 3.0$)", fontsize=11, fontweight='bold')
ax_comp.set_xticks(sites)
ax_comp.grid(True, linestyle='--', alpha=0.5)
ax_comp.legend(frameon=True, facecolor='white', framealpha=0.9, fontsize=9)

# ------------------------------------------------------------------------------
# BOTTOM: Circuit Depth Scaling Table (Contrained Width)
# ------------------------------------------------------------------------------
ax_table = fig.add_subplot(gs[1, :])
ax_table.axis('off')

table_data = ax_table.table(
    cellText=depth_df.values,
    colLabels=depth_df.columns,
    cellLoc='center',
    loc='center',
    colWidths=[0.12, 0.15, 0.18, 0.22, 0.25]  # Explicit column widths prevent rightward spill
)

table_data.auto_set_font_size(False)
table_data.set_fontsize(9.5)
table_data.scale(0.9, 1.8)

header_color = '#1f4e78'
zebra_color = '#f2f6fa'

for (row, col), cell in table_data.get_celld().items():
    cell.set_edgecolor('#d0d7de')
    if row == 0:
        cell.set_facecolor(header_color)
        cell.get_text().set_color('white')
        cell.get_text().set_weight('bold')
    else:
        cell.set_facecolor(zebra_color if row % 2 == 0 else '#ffffff')
        cell.get_text().set_color('#222222')

ax_table.set_title(
    "Circuit Depth Scaling vs. Trotter Step Size ($\Delta t$) at Target Time $t = 3.0$",
    fontsize=11,
    fontweight='bold',
    pad=12
)

# ==============================================================================
# 3. SAVE WITH BOUNDING BOX OVERRIDE
# ==============================================================================
plt.savefig(
    "fermi_hubbard_unified_summary_page.png",
    bbox_inches='tight',
    pad_inches=0.2,  # Adds explicit outer padding to PDF rendering
    dpi=300
)

plt.savefig(
    "fermi_hubbard_unified_summary_page.pdf",
    bbox_inches='tight',
    pad_inches=0.2,
    dpi=300
)

plt.show()

In [ ]:
def postselect(counts, n_particles):
    """Keep only shots with the correct total particle number."""
    kept = {b: p for b, p in counts.items() if b.count('1') == n_particles}
    survival = sum(kept.values()) / sum(counts.values())
    return kept, survival

qv = build_circuit(dt, 6, init_occ)
counts = qv.get_measurement()
kept, survival = postselect(counts, n_particles=L)
print(f"symmetry post-selection keeps {survival*100:.1f}% of shots")
print("(on the noiseless simulator this is 100% by construction --")
print(" on hardware it is the cheapest error-mitigation step available)")